In [1]:
import os
import numpy as np
import pandas as pd
import vsp
from pyDOE2 import lhs

In [2]:
TEMPLATE_VSP3 = "avanger_for_experiment.vsp3"  # 템플릿 파일명
OUTPUT_DIR = "generated_avenger_files"    # 생성 파일 저장 폴더

os.makedirs(OUTPUT_DIR, exist_ok=True)

## LHS Data Sampling

In [3]:
PARAM_BOUNDS = {
    "SweepAngle": (20.0, 40.0),           # deg
    "WingletLength": (0.5 * 12.4, 12.4),  # cm
    "CantAngle": (-90.0, 90.0),           # deg
    "TaperRatio": (0.3, 0.6)              # unitless
}

N_SAMPLES = 60 # 생성할 샘플 수

np.random.seed(42)
num_vars = len(PARAM_BOUNDS)
lhs_samples = lhs(num_vars, samples=N_SAMPLES)

# 스케일링
param_names = list(PARAM_BOUNDS.keys())
scaled_samples = {}
for i, name in enumerate(param_names):
    low, high = PARAM_BOUNDS[name]
    scaled_samples[name] = low + (high - low) * lhs_samples[:, i]

In [4]:
# 결과 저장

sampling_df = pd.DataFrame(scaled_samples)
sampling_df.index.name = "Index"
sampling_df.to_csv(os.path.join(OUTPUT_DIR, "winglet_lhs_samples.csv"))
print("Winglet 파라미터 LHS 샘플링 완료 및 CSV 저장!")

Winglet 파라미터 LHS 샘플링 완료 및 CSV 저장!


## 다양한 매개변수 값에 따른 Winglet 형상 파일 생성 및 저장

In [5]:
for idx in range(N_SAMPLES):
    # 샘플 값 불러오기
    sweep_angle = scaled_samples["SweepAngle"][idx]
    winglet_length = scaled_samples["WingletLength"][idx]
    cant_angle = scaled_samples["CantAngle"][idx]
    taper_ratio = scaled_samples["TaperRatio"][idx]

    # 템플릿 VSP 파일 로드
    vsp.ClearVSPModel()
    vsp.ReadVSPFile(TEMPLATE_VSP3)

    # MainWing Geom ID 찾기
    mainwing_id = vsp.FindGeom("MainWing", 0)

    # XSecSurf ID 가져오기
    xsec_surf_id = vsp.GetXSecSurf(mainwing_id, 0)

    # Section2, Section3 ID 가져오기
    section2_id = vsp.GetXSec(xsec_surf_id, 2)
    section3_id = vsp.GetXSec(xsec_surf_id, 3)

    # 파라미터 ID 추출 및 값 설정
    sweep_id = vsp.GetParm(section3_id, "Sweep", "XSec")
    span_id = vsp.GetParm(section3_id, "Span", "XSec")
    dihedral2_id = vsp.GetParm(section2_id, "Dihedral", "XSec")
    dihedral3_id = vsp.GetParm(section3_id, "Dihedral", "XSec")
    taper_id = vsp.GetParm(section3_id, "Taper", "XSec")

    vsp.SetParmVal(sweep_id, sweep_angle)
    vsp.SetParmVal(span_id, winglet_length)
    vsp.SetParmVal(dihedral2_id, cant_angle / 2)
    vsp.SetParmVal(dihedral3_id, cant_angle / 2)
    vsp.SetParmVal(taper_id, taper_ratio)

    vsp.Update()

    # 저장
    save_path = os.path.join(OUTPUT_DIR, f"sample_{idx:03d}.vsp3")
    vsp.WriteVSPFile(save_path)
    print(f"샘플 {idx} 저장 완료!")

샘플 0 저장 완료!
샘플 1 저장 완료!
샘플 2 저장 완료!
샘플 3 저장 완료!
샘플 4 저장 완료!
샘플 5 저장 완료!
샘플 6 저장 완료!
샘플 7 저장 완료!
샘플 8 저장 완료!
샘플 9 저장 완료!
샘플 10 저장 완료!
샘플 11 저장 완료!
샘플 12 저장 완료!
샘플 13 저장 완료!
샘플 14 저장 완료!
샘플 15 저장 완료!
샘플 16 저장 완료!
샘플 17 저장 완료!
샘플 18 저장 완료!
샘플 19 저장 완료!
샘플 20 저장 완료!
샘플 21 저장 완료!
샘플 22 저장 완료!
샘플 23 저장 완료!
샘플 24 저장 완료!
샘플 25 저장 완료!
샘플 26 저장 완료!
샘플 27 저장 완료!
샘플 28 저장 완료!
샘플 29 저장 완료!
샘플 30 저장 완료!
샘플 31 저장 완료!
샘플 32 저장 완료!
샘플 33 저장 완료!
샘플 34 저장 완료!
샘플 35 저장 완료!
샘플 36 저장 완료!
샘플 37 저장 완료!
샘플 38 저장 완료!
샘플 39 저장 완료!
샘플 40 저장 완료!
샘플 41 저장 완료!
샘플 42 저장 완료!
샘플 43 저장 완료!
샘플 44 저장 완료!
샘플 45 저장 완료!
샘플 46 저장 완료!
샘플 47 저장 완료!
샘플 48 저장 완료!
샘플 49 저장 완료!
샘플 50 저장 완료!
샘플 51 저장 완료!
샘플 52 저장 완료!
샘플 53 저장 완료!
샘플 54 저장 완료!
샘플 55 저장 완료!
샘플 56 저장 완료!
샘플 57 저장 완료!
샘플 58 저장 완료!
샘플 59 저장 완료!


## VSPAERO를 활용한 해석 자동화 시스템